<a href="https://colab.research.google.com/github/meem-5971/FlyRank-ML-/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meem-5971/FlyRank-ML-/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*



1. **Grain:** One row represents a unique `(client_hash_id, content_hash_id, query_hash)` combination aggregated over a mid-panel month (`2026-03-01` to `2026-03-31`).
2. **Tables Used:** `fact_content_query_90d.parquet` (or month-partitioned query slice) and `dim_content.parquet`.
3. **Time Window:** Mid-panel month `2026-03` (`2026-03-01` to `2026-03-31`).
4. **Target / Objective:** Unsupervised clustering of Search Console query-content pairs into intent and performance buckets (e.g., High-Intent Conversion, Low-CTR Long-Tail) to optimize content allocation.
5. **Deliberately Excluded:** Post-period analytics metrics or future performance data beyond `2026-03-31` (e.g., future conversions, next-month clicks).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [3]:
import os
import duckdb
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# Handle HF Token access for Hugging Face gated dataset
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.getenv('HF_TOKEN', '')

assert HF_TOKEN, "Please set your HF_TOKEN in Colab Secrets or as an environment variable."

# Initialize DuckDB Connection & Attach Hugging Face Secret
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# Warehouse remote paths
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_QUERY = f"{REL}/fact_content_query_90d.parquet"
FACT_DAILY = f"{REL}/fact_content_daily_performance/**/*.parquet"
DIM_CONTENT = f"{REL}/dim_content.parquet"

print("DuckDB successfully authenticated and connected to Hugging Face warehouse.")

DuckDB successfully authenticated and connected to Hugging Face warehouse.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [9]:
# --- Fact 1: Prove Grain Uniqueness ---
# Query 1: Verify that (client_hash_id, content_hash_id, query_hash_id) uniquely identifies a row
fact_1_sql = f"""
SELECT
    client_hash_id,
    content_hash_id,
    query_hash_id,
    COUNT(*) AS row_count
FROM read_parquet('{FACT_QUERY}')
GROUP BY client_hash_id, content_hash_id, query_hash_id
HAVING COUNT(*) > 1
LIMIT 10;
"""
df_fact_1 = con.sql(fact_1_sql).df()
print("=== Fact 1: Grain Duplicate Check ===")
print(f"Duplicates found: {len(df_fact_1)} (Expected: 0)")
display(df_fact_1)


# --- Fact 2: Slice Row Count & Date Span ---
# Query 2: Prove total row volume and date range for the mid-panel slice
fact_2_sql = f"""
SELECT
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id) AS unique_clients,
    COUNT(DISTINCT content_hash_id) AS unique_content_items
FROM read_parquet('{FACT_DAILY}')
WHERE month = '2026-03';
"""
df_fact_2 = con.sql(fact_2_sql).df()
print("\n=== Fact 2: Row Count & Date Span (2026-03) ===")
display(df_fact_2)


# --- Fact 3: Availability Filtering ---
# Query 3: Apply IS TRUE filter on valid record flags (using gsc_impressions and gsc_clicks)
fact_3_sql = f"""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN is_valid IS TRUE THEN 1 ELSE 0 END) AS valid_surviving_rows,
    ROUND((SUM(CASE WHEN is_valid IS TRUE THEN 1 ELSE 0 END) * 100.0 / COUNT(*)), 2) AS survival_rate_pct
FROM (
    SELECT
        (gsc_impressions > 0 AND gsc_clicks >= 0) AS is_valid
    FROM read_parquet('{FACT_DAILY}')
    WHERE month = '2026-03'
);
"""
df_fact_3 = con.sql(fact_3_sql).df()
print("\n=== Fact 3: Availability Survival Rate ===")
display(df_fact_3)

=== Fact 1: Grain Duplicate Check ===
Duplicates found: 0 (Expected: 0)


,client_hash_id,content_hash_id,query_hash_id,row_count


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== Fact 2: Row Count & Date Span (2026-03) ===


,min_date,max_date,total_rows,unique_clients,unique_content_items
0,2026-03-01,2026-03-31,9841378,55,331437


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== Fact 3: Availability Survival Rate ===


,total_rows,valid_surviving_rows,survival_rate_pct
0,9841378,3611061.0,36.69


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [11]:
# Build Feature Frame from 2026-03 with 5 knowable features + 1 leaked feature
feature_frame_sql = f"""
WITH base_features AS (
    SELECT
        client_hash_id,
        content_hash_id,
        query_hash_id,
        impressions_last30 AS total_impressions,
        clicks_last30 AS total_clicks,
        CAST(clicks_last30 AS FLOAT) / NULLIF(impressions_last30, 0) AS historical_ctr,
        avg_position_last30 AS avg_position,
        LENGTH(query_hash_id) AS query_length_proxy
    FROM read_parquet('{FACT_QUERY}')
    WHERE impressions_last30 > 10
    LIMIT 5000
),
leaked_label AS (
    -- TRAP: Injecting future clicks from next month (2026-04)
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS future_next_month_clicks
    FROM read_parquet('{FACT_DAILY}')
    WHERE month = '2026-04'
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    f.total_impressions,
    f.total_clicks,
    f.historical_ctr,
    f.avg_position,
    f.query_length_proxy,
    COALESCE(l.future_next_month_clicks, 0) AS future_next_month_clicks -- LEAKED FEATURE
FROM base_features f
LEFT JOIN leaked_label l
    ON f.client_hash_id = l.client_hash_id
   AND f.content_hash_id = l.content_hash_id;
"""

df_features = con.sql(feature_frame_sql).df().fillna(0)

print("Feature Frame preview:")
display(df_features.head())

# Print "Available When?" Justifications
print("\n" + "="*80)
print("FEATURE AVAILABILITY JUSTIFICATIONS:")
print("1. total_impressions: Knowable at the decision moment because it aggregates logs within 2026-03.")
print("2. total_clicks: Knowable at the decision moment because it reflects past clicks up to 2026-03-31.")
print("3. historical_ctr: Knowable at the decision moment because it derives strictly from historical 2026-03 metrics.")
print("4. avg_position: Knowable at the decision moment because rank positions were recorded prior to window close.")
print("5. query_length_proxy: Knowable at the decision moment because string characteristics are static query metadata.")
print("="*80 + "\n")

# --- TRAP EXECUTION ---
# 1. Evaluate KMeans with Leaked Feature
features_leaked = ['total_impressions', 'total_clicks', 'historical_ctr', 'avg_position', 'query_length_proxy', 'future_next_month_clicks']
X_leaked = StandardScaler().fit_transform(df_features[features_leaked])

kmeans_leaked = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X_leaked)
score_leaked = silhouette_score(X_leaked, kmeans_leaked.labels_)
print(f"⚠️ Silhouette Score WITH Target Leakage: {score_leaked:.4f} (Artificially High/Inflated)")

# 2. Drop Leaked Feature & Report Honest Metric
features_honest = ['total_impressions', 'total_clicks', 'historical_ctr', 'avg_position', 'query_length_proxy']
X_honest = StandardScaler().fit_transform(df_features[features_honest])

kmeans_honest = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X_honest)
score_honest = silhouette_score(X_honest, kmeans_honest.labels_)
print(f"✅ Silhouette Score WITHOUT Leakage (Honest): {score_honest:.4f}")

# Remove the leaked column from dataframe to keep feature set clean
df_features_clean = df_features[features_honest]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature Frame preview:


,total_impressions,total_clicks,historical_ctr,avg_position,query_length_proxy,future_next_month_clicks
0,11,0,0.0,24.272727,22,1.0
1,29,0,0.0,10.379310,22,0.0
2,11,0,0.0,8.818182,22,0.0
3,48,0,0.0,52.604167,22,5.0
4,119,0,0.0,7.798319,22,5.0



FEATURE AVAILABILITY JUSTIFICATIONS:
1. total_impressions: Knowable at the decision moment because it aggregates logs within 2026-03.
2. total_clicks: Knowable at the decision moment because it reflects past clicks up to 2026-03-31.
3. historical_ctr: Knowable at the decision moment because it derives strictly from historical 2026-03 metrics.
4. avg_position: Knowable at the decision moment because rank positions were recorded prior to window close.
5. query_length_proxy: Knowable at the decision moment because string characteristics are static query metadata.

⚠️ Silhouette Score WITH Target Leakage: 0.5311 (Artificially High/Inflated)
✅ Silhouette Score WITHOUT Leakage (Honest): 0.6104


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.